<a href="https://colab.research.google.com/github/djwillichile/geoia-bloom-huasco/blob/main/notebooks/bloom_desierto_florido_huasco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# 🌸 Bloom del Desierto Florido — Cuenca del Huasco

**Análisis de la dinámica espacio-temporal del Desierto Florido (Atacama, Chile)**
usando series temporales de NDVI MODIS MOD13Q1 · Google Earth Engine

---

**Área de estudio:** Cuenca del río Huasco, Región de Atacama, Chile  
**Período:** 2015–2019  
**Resolución:** 250 m · 16 días  

</div>

## 0) Instalación e inicialización de Earth Engine

En esta sección se instalan los paquetes necesarios y se inicializa la sesión de Earth Engine.

> La primera vez que ejecutes el notebook tendrás que autenticarte.


In [ ]:
# Instalar paquetes solo si no están instalados
import importlib.util
import os

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    if importlib.util.find_spec(package) is None:
        print(f"Instalando {pip_name}...")
        os.system(f"pip -q install {pip_name}")
    else:
        print(f"✅ {pip_name} ya está instalado")

install_if_missing("geemap")
install_if_missing("ee", "earthengine-api")

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# Inicialización de Earth Engine
from google.colab import userdata
# projectGEE = 'my-project'
projectGEE = userdata.get('projectGEE')

try:
    ee.Initialize(project=projectGEE)
    print("✅ Google Earth Engine inicializado")
except Exception as e:
    print("Iniciando autenticación...")
    ee.Authenticate()
    ee.Initialize(project=projectGEE)
    print("✅ Google Earth Engine inicializado tras autenticación")

✅ geemap ya está instalado
✅ earthengine-api ya está instalado
✅ Google Earth Engine inicializado


## 1) Definición del área de estudio

En esta parte se construye el area de interes mediente lsa geometrías de trabajo para la **cuenca del rio Huasco**.

### Componentes espaciales usados
- **GAUL**: para ubicar la provincia del Huasco
- **HydroBASINS**: para delimitar la cuenca hidrográfica


> Aunque se visualiza un AOI más amplio, las métricas se calculan **dentro de la geometría de la cuenca**.



In [ ]:
# -----------------------------
# Geometrías
# -----------------------------

# Provincia
gaul = ee.FeatureCollection("FAO/GAUL/2015/level2")

huasco_prov = (gaul
  .filter(ee.Filter.eq('ADM0_NAME', 'Chile'))
  .filter(ee.Filter.eq('ADM2_NAME', 'Huasco'))
  .geometry()
)

outlet = ee.Geometry.Point([-71.17, -28.47])

basins = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_6')
huasco_basin = ee.Feature(basins.filterBounds(outlet).first()).geometry()

# Area de interés AOI (box con buffer)
AOI = huasco_basin.union(huasco_prov).bounds().buffer(5000).bounds()

print("✅ Geometrias cargadas")


✅ Geometrias cargadas


In [ ]:
# =============================
# Previsuacización del area de interés
# =============================
import geemap

# Habilitar widgets ipyleaflet en Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Definir función para agregar capas de referencia al mapa
def add_reference_layers(Map, zoom=8):
    # Centrar mapa
    Map.centerObject(AOI, zoom)

    # Contornos vectoriales
    Map.addLayer(ee.Image().paint(AOI, 1, 2), {"palette": ["blue"]}, "AOI")
    Map.addLayer(ee.Image().paint(huasco_basin, 1, 1.8), {"palette": ["red"]}, "Huasco basin")
    Map.addLayer(ee.Image().paint(huasco_prov, 1, 1.5), {"palette": ["darkblue"]}, "Huasco province")

    # Punto de salida
    Map.addLayer(outlet, {"color": "black"}, "Outlet")

# --- visual context ---
# Elevation
elev = ee.Image("USGS/SRTMGL1_003").clip(AOI)

# Hillshade
hillshade = ee.Terrain.hillshade(elev)

# Paletas de color
elevVis = {
    "min": 0,
    "max": 5500,
    "palette": [
        "#0d4f8b", "#2e8b57", "#a6d96a", "#fee08b",
        "#fdae61", "#d73027", "#7f3b08", "#f5f5f5",
    ],
    "opacity": 0.85
}

rgb = elev.visualize(**elevVis)

shaded = ee.Image.rgb(
    rgb.select(0).multiply(hillshade.divide(255)),
    rgb.select(1).multiply(hillshade.divide(255)),
    rgb.select(2).multiply(hillshade.divide(255))
)

# Generar mapa base
Map = geemap.Map()
Map.add_basemap("HYBRID")

# Map.addLayer(elev, elevVis, "Elevation (m)")
# Map.addLayer(hillshade, {"min": 0, "max": 255, "opacity": 0.3}, "Hillshade")

Map.addLayer(shaded, {}, "Topography shaded")

# Capas vectoriales de contexto
add_reference_layers(Map)

print("✅ Mapa interactivo desplegado")

Map

✅ Mapa interactivo desplegado


Map(center=[-28.859256987602333, -70.63855948678793], controls=(WidgetControl(options=['position', 'transparen…

## 2) Análisis de bloom

> Agrega aquí las celdas de análisis.